# P2: SAM 2.1 Leaf Crop Check (50 images)
Run SAM 2.1 Hiera Tiny on 25 PlantVillage + 25 PlantDoc tomato images and check that each crop contains the whole leaf.
Rule from the proposal: **≥ 40/50 pass → keep SAM, otherwise fall back to a center crop.**
Runs in Colab (GPU) or locally in VS Code (Mac uses MPS or CPU).

In [ ]:
# Install SAM 2 and load the frozen SAM 2.1 Hiera Tiny model
%pip install -q git+https://github.com/facebookresearch/sam2.git

import random, time, io
from zipfile import ZipFile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from huggingface_hub import hf_hub_download
from sam2.sam2_image_predictor import SAM2ImagePredictor

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", device)
predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2.1-hiera-tiny", device=device)
random.seed(539)

In [ ]:
# 25 PlantVillage images (same download and class names as P1)
download = hf_hub_download(repo_id="mohanty/PlantVillage", filename="data.zip", repo_type="dataset")

classes = ["Tomato___healthy", "Tomato___Early_blight", "Tomato___Late_blight"]
zip_file = ZipFile(download)
pv_files = [f for f in zip_file.namelist()
            if "/color/" in f and f.split("/")[-2] in classes and f.lower().endswith((".jpg", ".jpeg", ".png"))]

pv_samples = []
for f in random.sample(pv_files, 25):
    image = Image.open(io.BytesIO(zip_file.read(f))).convert("RGB")
    pv_samples.append(("PlantVillage", f.split("/")[-2], image))

print("PlantVillage images:", len(pv_samples))

In [ ]:
# 25 PlantDoc images from the TEST split (external data, never used for training)
!git clone -q https://github.com/pratikkayal/PlantDoc-Dataset.git

pd_folders = ["Tomato leaf", "Tomato Early blight leaf", "Tomato leaf late blight"]
pd_files = []
for folder in pd_folders:
    for f in Path("PlantDoc-Dataset/test", folder).iterdir():
        if f.suffix.lower() in (".jpg", ".jpeg", ".png"):
            pd_files.append((folder, f))
    print(folder, len(list(Path("PlantDoc-Dataset/test", folder).iterdir())))

pd_samples = [("PlantDoc", folder, Image.open(f).convert("RGB")) for folder, f in random.sample(pd_files, 25)]
samples = pv_samples + pd_samples
print("Total images:", len(samples))

In [ ]:
# Leaf crop: one positive point at the image center -> best mask -> padded SQUARE box -> 224 x 224
def crop_leaf(image, pad=0.10):
    w, h = image.size
    predictor.set_image(np.array(image))
    masks, scores, _ = predictor.predict(point_coords=np.array([[w / 2, h / 2]]),
                                         point_labels=np.array([1]),
                                         multimask_output=True)
    mask = masks[np.argmax(scores)].astype(bool)

    ys, xs = np.where(mask)
    if len(xs) == 0:  # no mask found -> keep whole image
        return mask, image.resize((224, 224))

    # Square box around the mask center (no stretching when resized), kept inside the image
    cx, cy = (xs.min() + xs.max()) / 2, (ys.min() + ys.max()) / 2
    side = min(max(xs.max() - xs.min(), ys.max() - ys.min()) * (1 + 2 * pad), w, h)
    left = min(max(cx - side / 2, 0), w - side)
    top = min(max(cy - side / 2, 0), h - side)
    box = (int(left), int(top), int(left + side), int(top + side))
    return mask, image.crop(box).resize((224, 224))

In [ ]:
# Run on all 50 images and save a grid: original | mask | crop
results = []
start = time.time()
for source, label, image in samples:
    mask, crop = crop_leaf(image)
    results.append((source, label, image, mask, crop))
print(f"Seconds per image: {(time.time() - start) / len(samples):.2f}")

fig, axes = plt.subplots(len(results), 3, figsize=(9, 3 * len(results)))
for i, (source, label, image, mask, crop) in enumerate(results):
    axes[i, 0].imshow(image);  axes[i, 0].set_title(f"#{i} {source} | {label}", fontsize=8)
    axes[i, 1].imshow(image);  axes[i, 1].imshow(mask, alpha=0.5, cmap="jet")
    axes[i, 2].imshow(crop);   axes[i, 2].set_title(f"crop {crop.size}", fontsize=8)
    for ax in axes[i]: ax.axis("off")
plt.tight_layout()
plt.savefig("crop_check.png", dpi=80)
plt.show()

In [ ]:
# Manual check: look at the grid and set 0 for every crop that does NOT contain the whole leaf
passed = [1] * 50
# Example: passed[3] = 0

pv_pass, pd_pass = sum(passed[:25]), sum(passed[25:])
total = pv_pass + pd_pass
print(f"PlantVillage: {pv_pass}/25   PlantDoc: {pd_pass}/25   Total: {total}/50")
print("Decision:", "keep SAM" if total >= 40 else "fall back to center crop")